In [1]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

base_dir = os.path.dirname(os.getcwd())
data_dir = os.path.join(base_dir, "data")

clean_file = os.path.join(data_dir, "epl_team_matches_clean.csv")
df = pd.read_csv(clean_file, parse_dates=["Date"])

print("Data shape:", df.shape)
df.head()


Data shape: (1520, 18)


,Date,team,opponent,goals_for,goals_against,shots_for,shots_against,sot_for,sot_against,corners_for,corners_against,total_goals,over25,is_home,goal_diff,is_win,is_draw,is_loss
0,2022-08-05,Arsenal,Crystal Palace,2,0,10,10,2,2,5,3,2,0,0,2,1,0,0
1,2022-08-13,Arsenal,Leicester,4,2,19,6,7,2,6,2,6,1,1,2,1,0,0
2,2022-08-20,Arsenal,Bournemouth,3,0,14,6,6,1,4,3,3,1,0,3,1,0,0
3,2022-08-27,Arsenal,Fulham,2,1,22,11,8,3,9,2,3,1,1,1,1,0,0
4,2022-08-31,Arsenal,Aston Villa,2,1,22,4,9,3,10,3,3,1,1,1,1,0,0


In [2]:
# Features and target (exclude goals_for and goals_against)

feature_cols = [
    "shots_for",
    "shots_against",
    "sot_for",
    "sot_against",
    "corners_for",
    "corners_against",
    "is_home",
]

X = df[feature_cols].copy()
y = df["over25"].copy()

print("X shape:", X.shape)
print("y value counts:")
print(y.value_counts())



X shape: (1520, 7)
y value counts:
over25
1    892
0    628
Name: count, dtype: int64


In [3]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


Train: (1216, 7) Test: (304, 7)


In [4]:
# Train logistic regression classifier
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

y_pred = log_reg.predict(X_test_scaled)
y_prob = log_reg.predict_proba(X_test_scaled)[:, 1]


In [5]:
# Evaluation metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_prob)

print("Accuracy:", round(acc, 3))
print("Precision:", round(prec, 3))
print("Recall:", round(rec, 3))
print("F1 score:", round(f1, 3))
print("ROC-AUC:", round(roc, 3))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))


Accuracy: 0.714
Precision: 0.733
Recall: 0.803
F1 score: 0.767
ROC-AUC: 0.769

Confusion matrix:
[[ 74  52]
 [ 35 143]]


In [6]:
# Feature importance for logistic regression (absolute coefficients)
importance = pd.Series(
    np.abs(log_reg.coef_[0]),
    index=feature_cols
).sort_values(ascending=False)

importance


sot_for            1.198381
sot_against        1.057775
shots_for          0.357675
shots_against      0.281542
corners_against    0.107004
corners_for        0.092071
is_home            0.022956
dtype: float64

## This tells me which stats influence over‑2.5 goals most (e.g., high shots_for, high total corners, etc.)

In [7]:
import joblib

models_dir = os.path.join(base_dir, "models")
os.makedirs(models_dir, exist_ok=True)

joblib.dump(log_reg, os.path.join(models_dir, "model_over25_logreg.pkl"))
joblib.dump(scaler, os.path.join(models_dir, "scaler_over25.pkl"))

print("Saved model and scaler in", models_dir)


Saved model and scaler in C:\Users\Mohsin\football_capstone\models
